In [5]:
# ============================================
# AUTO-RUN CTR Ensemble Training (Notebook Safe)
# ============================================

import random
import numpy as np
import pandas as pd
from datasets import load_dataset

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTEN, RandomOverSampler

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import scipy.sparse as sp
import warnings
warnings.filterwarnings("ignore")


# ----------------------------------------
# STEP 1 — STREAM + BALANCE
# ----------------------------------------
def stream_balanced_subset(train_size_per_class=5000, max_minority=5000, seed=42):
    print("⏳ Streaming from HuggingFace...")

    ds = load_dataset("BadrAbu/CTR_Prediction", split="train", streaming=True)

    pos, neg = [], []
    random.seed(seed)

    for row in ds:
        row = dict(row)
        if row["click"] == 1 and len(pos) < train_size_per_class:
            pos.append(row)
        elif row["click"] == 0 and len(neg) < train_size_per_class:
            neg.append(row)

        if len(pos) >= train_size_per_class and len(neg) >= train_size_per_class:
            break

    df = pd.DataFrame(pos + neg)
    print(f"✅ Initial subset: {df.shape}")

    X = df.drop(columns=["click"])
    y = df["click"]

    # Balance
    rus = RandomUnderSampler(
        sampling_strategy={0: max_minority, 1: max_minority},
        random_state=seed
    )
    X_step, y_step = rus.fit_resample(X, y)

    try:
        sm = SMOTEN(random_state=seed)
        X_bal, y_bal = sm.fit_resample(X_step, y_step)
    except:
        ros = RandomOverSampler(random_state=seed)
        X_bal, y_bal = ros.fit_resample(X_step, y_step)

    print(f"🎯 Balanced final shape: {X_bal.shape}")
    return X_bal, y_bal


# ----------------------------------------
# STEP 2 — ENCODING
# ----------------------------------------
def encode_features(X):
    for col in X.select_dtypes(include=["object"]).columns:
        X[col] = X[col].astype("category")

    cat_cols = X.select_dtypes(include=["category", "object"]).columns.tolist()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

    transformer = ColumnTransformer([
        ("cat", ohe, cat_cols),
        ("num", "passthrough", num_cols)
    ])

    print(f"📦 Encoding — {len(cat_cols)} categorical, {len(num_cols)} numeric")

    return transformer


# ----------------------------------------
# STEP 3 — TRAIN MODELS
# ----------------------------------------
def train_models(X_train, y_train, X_val, y_val):
    if not sp.isspmatrix_csr(X_train):
        X_train = sp.csr_matrix(X_train)
        X_val = sp.csr_matrix(X_val)

    results = {}

    # CatBoost
    print("\n🚀 Training CatBoost...")
    cb = CatBoostClassifier(
        iterations=80,
        learning_rate=0.1,
        depth=6,
        verbose=False
    )
    cb.fit(X_train, y_train)
    cb_pred = cb.predict_proba(X_val)[:, 1]
    results["catboost_auc"] = roc_auc_score(y_val, cb_pred)
    print("✔ CatBoost AUC:", results["catboost_auc"])

    # LightGBM
    print("\n🚀 Training LightGBM...")
    lgb = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1
    )
    lgb.fit(X_train, y_train)
    lgb_pred = lgb.predict_proba(X_val)[:, 1]
    results["lightgbm_auc"] = roc_auc_score(y_val, lgb_pred)
    print("✔ LightGBM AUC:", results["lightgbm_auc"])

    # XGBoost
    print("\n🚀 Training XGBoost...")
    xgb = XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        eval_metric="auc",
        tree_method="hist"
    )
    xgb.fit(X_train, y_train)
    xgb_pred = xgb.predict_proba(X_val)[:, 1]
    results["xgboost_auc"] = roc_auc_score(y_val, xgb_pred)
    print("✔ XGBoost AUC:", results["xgboost_auc"])

    # Ensemble
    ensemble = (cb_pred + lgb_pred + xgb_pred) / 3
    results["ensemble_auc"] = roc_auc_score(y_val, ensemble)
    print("\n🎯 Ensemble AUC:", results["ensemble_auc"])

    return results


# ----------------------------------------
# AUTO-RUN PIPELINE
# ----------------------------------------

print("🔥 AUTO-RUN PIPELINE STARTED")

# Step 1
X_bal, y_bal = stream_balanced_subset(
    train_size_per_class=4000,
    max_minority=4000
)

# Step 2
encoder = encode_features(X_bal.copy())
X_train, X_val, y_train, y_val = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
)

X_train_enc = encoder.fit_transform(X_train)
X_val_enc = encoder.transform(X_val)

print("🔧 Encoded shapes:", X_train_enc.shape, X_val_enc.shape)

# Step 3
results = train_models(X_train_enc, y_train, X_val_enc, y_val)

print("\n==============================")
print("🏁 FINAL RESULTS")
print("==============================")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

print("\n✅ All Done!")


🔥 AUTO-RUN PIPELINE STARTED
⏳ Streaming from HuggingFace...
✅ Initial subset: (8000, 24)
🎯 Balanced final shape: (8000, 23)
📦 Encoding — 9 categorical, 14 numeric
🔧 Encoded shapes: (6400, 7488) (1600, 7488)

🚀 Training CatBoost...
✔ CatBoost AUC: 0.9294359375000001

🚀 Training LightGBM...
[LightGBM] [Info] Number of positive: 3200, number of negative: 3200
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000904 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 971
[LightGBM] [Info] Number of data points in the train set: 6400, number of used features: 172
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
✔ LightGBM AUC: 0.92907109375

🚀 Training XGBoost...
✔ XGBoost AUC: 0.9351085937499999

🎯 Ensemble AUC: 0.93555078125

🏁 FINAL RESULTS
catboost_auc: 0.9294
lightgbm_auc: 0.9291
xgboost_auc: 0.9351
ensemble_auc: